# 多阶段评分筛选模型

**参考论文**: Yanrui Li (2022), "How to make machine select stocks like fund managers?"  
**核心思想**: 模拟基金经理的多阶段筛选流程: 动量评分 -> 波动率过滤 -> 综合排名  
**方法**: 纯规则驱动 (无ML), 通过层层筛选从20只A股中选出最优组合

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 策略原理

### 三阶段筛选模型

**Stage 1: 动量评分**

计算多周期动量加权得分:
$$S_{\text{mom},i} = 0.3 \cdot \text{rank}(r_{20d}) + 0.3 \cdot \text{rank}(r_{60d}) + 0.4 \cdot \text{rank}(r_{120d})$$

筛选条件: $S_{\text{mom},i} \geq \text{median}(S_{\text{mom}})$，保留前50%

**Stage 2: 波动率过滤**

计算20日年化波动率:
$$\sigma_i = \text{std}(r_{i,t}) \times \sqrt{244}$$

筛选条件: $\sigma_i \leq Q_{75\%}(\sigma)$，剔除波动率最高的25%

**Stage 3: 综合排名**

综合得分:
$$S_{\text{total},i} = 0.5 \cdot S_{\text{mom},i} + 0.3 \cdot (1 - \text{rank}(\sigma_i)) + 0.2 \cdot \text{rank}(\text{turnover}_i)$$

选择Top-N (如Top-5) 等权配置

### 漏斗筛选流程
$$\text{全部20只} \xrightarrow{\text{Stage 1}} \text{10只} \xrightarrow{\text{Stage 2}} \text{7~8只} \xrightarrow{\text{Stage 3}} \text{Top 5}$$

In [ ]:
# Cell 3: 动画 - 筛选漏斗可视化
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

def animate_screening_funnel():
    """动画展示股票池逐级筛选的漏斗过程"""
    stages = ['初始股票池\n(20只)', 'Stage 1: 动量筛选\n(10只)',
              'Stage 2: 波动率过滤\n(7只)', 'Stage 3: 综合Top-5\n(5只)']
    counts = [20, 10, 7, 5]
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

    # 模拟每只股票的得分
    stock_names = [f'Stock_{i+1:02d}' for i in range(20)]
    mom_scores = np.random.uniform(0, 1, 20)
    vol_scores = np.random.uniform(0.15, 0.45, 20)
    composite_scores = np.random.uniform(0, 1, 20)

    # 排序筛选
    stage1_idx = np.argsort(mom_scores)[-10:]  # Top10 by momentum
    stage2_idx = [i for i in stage1_idx if vol_scores[i] <= np.percentile(vol_scores[stage1_idx], 75)]
    if len(stage2_idx) > 7:
        stage2_idx = sorted(stage2_idx, key=lambda i: composite_scores[i], reverse=True)[:7]
    stage3_idx = sorted(stage2_idx, key=lambda i: composite_scores[i], reverse=True)[:5]

    frames = []
    all_stages = [list(range(20)), list(stage1_idx), stage2_idx, stage3_idx]

    for s in range(4):
        active = all_stages[s]
        marker_colors = ['green' if i in active else 'lightgray' for i in range(20)]
        marker_sizes = [15 if i in active else 8 for i in range(20)]

        # 散点: 动量 vs 波动率
        frames.append(go.Frame(
            data=[
                go.Scatter(
                    x=vol_scores, y=mom_scores,
                    mode='markers+text',
                    text=[f'{stock_names[i]}' if i in active else '' for i in range(20)],
                    textposition='top center',
                    marker=dict(color=marker_colors, size=marker_sizes,
                               line=dict(width=1, color='black')),
                    hovertext=[f'{stock_names[i]}<br>动量={mom_scores[i]:.2f}<br>波动={vol_scores[i]:.2f}'
                               for i in range(20)],
                ),
                # 漏斗标注
                go.Scatter(
                    x=[0.5], y=[1.05],
                    mode='text',
                    text=[f'{stages[s]}: {len(active)}只通过'],
                    textfont=dict(size=16, color=colors[s]),
                    showlegend=False,
                ),
            ],
            name=stages[s].split('\n')[0]
        ))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title='多阶段筛选漏斗: 股票池逐级缩小',
        xaxis_title='波动率', yaxis_title='动量得分',
        height=550, width=800,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 下一阶段', method='animate',
                 args=[None, {'frame': {'duration': 1500}, 'transition': {'duration': 500}}]),
        ])],
        sliders=[dict(steps=[
            dict(args=[[f.name]], label=f.name, method='animate') for f in frames
        ])],
    )
    return fig

fig = animate_screening_funnel()
fig.show()

In [ ]:
# Cell 4: 导入依赖
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from shared.data_fetcher import get_stock_daily, get_index_daily
from shared.constants import (DEFAULT_START, DEFAULT_END, RISK_FREE_RATE,
                               TRADING_DAYS_PER_YEAR, INITIAL_CAPITAL)
from shared.metrics import summary_table
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)

np.random.seed(42)
print('All imports loaded successfully.')

In [ ]:
# Cell 5: 下载20只代表性A股数据

stock_pool = {
    '贵州茅台': '600519', '五粮液': '000858', '招商银行': '600036',
    '中国平安': '601318', '工商银行': '601398', '海尔智家': '600690',
    '恒瑞医药': '600276', '药明康德': '603259', '宁德时代': '300750',
    '比亚迪': '002594', '隆基绿能': '601012', '海康威视': '002415',
    '美的集团': '000333', '格力电器': '000651', '万科A': '000002',
    '中信证券': '600030', '三一重工': '600031', '长江电力': '600900',
    '中国中免': '601888', '紫金矿业': '601899',
}

raw_data = {}
for name, code in stock_pool.items():
    df = get_stock_daily(code, DEFAULT_START, DEFAULT_END)
    if df is not None and len(df) > 200 and 'close' in df.columns:
        raw_data[name] = df
    else:
        print(f'  {name}({code}) 下载失败')

# 收盘价矩阵
close_dict = {name: df['close'] for name, df in raw_data.items()}
prices_df = pd.DataFrame(close_dict).sort_index().ffill().dropna()

# 换手率矩阵 (如有)
turnover_dict = {}
for name, df in raw_data.items():
    if 'turnover' in df.columns:
        turnover_dict[name] = df['turnover']
turnover_df = pd.DataFrame(turnover_dict).reindex(prices_df.index).ffill().fillna(0)

# 基准
bench_df = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
bench_close = bench_df['close'] if len(bench_df) > 0 else None

print(f'\n成功下载 {len(raw_data)} 只股票')
print(f'数据区间: {prices_df.index[0].date()} ~ {prices_df.index[-1].date()}')
print(f'交易日: {prices_df.shape[0]}')

In [ ]:
# Cell 6: 三阶段筛选函数

def stage1_momentum_score(prices_df, date_idx, lookback=120):
    """
    Stage 1: 动量评分
    加权动量 = 0.3*rank(20d) + 0.3*rank(60d) + 0.4*rank(120d)
    保留前50%
    """
    close = prices_df.iloc[max(0, date_idx-lookback):date_idx]
    if len(close) < 60:
        return list(prices_df.columns), pd.Series(0.5, index=prices_df.columns)

    scores = pd.Series(0.0, index=prices_df.columns)
    for name in prices_df.columns:
        c = close[name].dropna()
        if len(c) < 20:
            continue
        mom_20 = c.iloc[-1] / c.iloc[-min(20, len(c))] - 1
        mom_60 = c.iloc[-1] / c.iloc[-min(60, len(c))] - 1 if len(c) >= 60 else mom_20
        mom_120 = c.iloc[-1] / c.iloc[-min(120, len(c))] - 1 if len(c) >= 120 else mom_60
        scores[name] = 0.3 * mom_20 + 0.3 * mom_60 + 0.4 * mom_120

    # 排名 -> [0,1]
    rank_scores = scores.rank(pct=True)
    median_score = rank_scores.median()
    passed = rank_scores[rank_scores >= median_score].index.tolist()
    return passed, rank_scores


def stage2_volatility_filter(prices_df, candidates, date_idx, lookback=20):
    """
    Stage 2: 波动率过滤
    剔除波动率最高的25%
    """
    close = prices_df[candidates].iloc[max(0, date_idx-lookback):date_idx]
    if len(close) < 10:
        return candidates, pd.Series(0.0, index=candidates)

    vol = close.pct_change().std() * np.sqrt(TRADING_DAYS_PER_YEAR)
    threshold = vol.quantile(0.75)
    passed = vol[vol <= threshold].index.tolist()
    return passed, vol


def stage3_composite_ranking(mom_scores, vol_scores, turnover_scores, candidates, top_n=5):
    """
    Stage 3: 综合排名
    Score = 0.5*momentum_rank + 0.3*(1-vol_rank) + 0.2*turnover_rank
    选Top-N
    """
    composite = pd.Series(0.0, index=candidates)
    for name in candidates:
        m = mom_scores.get(name, 0.5) if name in mom_scores.index else 0.5
        v = vol_scores.get(name, 0.5) if name in vol_scores.index else 0.5
        t = turnover_scores.get(name, 0.5) if name in turnover_scores.index else 0.5

        # 波动率越低越好
        vol_rank = 1 - v / (vol_scores.max() + 1e-10) if vol_scores.max() > 0 else 0.5
        turn_rank = t / (turnover_scores.max() + 1e-10) if turnover_scores.max() > 0 else 0.5

        composite[name] = 0.5 * m + 0.3 * vol_rank + 0.2 * turn_rank

    return composite.nlargest(top_n).index.tolist(), composite


print('三阶段筛选函数定义完成')

In [ ]:
# Cell 7: 滚动筛选回测

returns_df = prices_df.pct_change().dropna()
stock_names = list(prices_df.columns)

lookback = 120
rebalance_freq = 20   # 月度调仓
top_n = 5             # 选择前5只

dates = returns_df.index
portfolio_returns = pd.Series(0.0, index=dates)
current_holdings = stock_names[:top_n]  # 初始等权
selection_history = []
funnel_history = []  # 记录漏斗过程

for i in range(lookback, len(dates)):
    # 持仓收益 (等权)
    valid_holdings = [s for s in current_holdings if s in returns_df.columns]
    if valid_holdings:
        portfolio_returns.iloc[i] = returns_df.iloc[i][valid_holdings].mean()

    # 调仓日
    if (i - lookback) % rebalance_freq == 0:
        # Stage 1: 动量
        stage1_passed, mom_scores = stage1_momentum_score(prices_df, i, lookback)

        # Stage 2: 波动率
        stage2_passed, vol_scores = stage2_volatility_filter(prices_df, stage1_passed, i)

        # Stage 3: 综合排名
        # 获取换手率排名
        if len(turnover_df) > 0:
            recent_turn = turnover_df.iloc[max(0,i-20):i].mean()
        else:
            recent_turn = pd.Series(0.5, index=stage2_passed)

        selected, comp_scores = stage3_composite_ranking(
            mom_scores, vol_scores, recent_turn, stage2_passed, top_n
        )

        old_holdings = set(current_holdings)
        current_holdings = selected if selected else current_holdings
        new_holdings = set(current_holdings)

        # 调仓成本
        turnover_pct = len(old_holdings.symmetric_difference(new_holdings)) / (2 * top_n)
        portfolio_returns.iloc[i] -= turnover_pct * 0.003  # 换手成本

        selection_history.append({
            'date': dates[i],
            'stage1_count': len(stage1_passed),
            'stage2_count': len(stage2_passed),
            'selected': current_holdings,
            'turnover': turnover_pct,
        })

# 截取有效区间
portfolio_returns = portfolio_returns.iloc[lookback:]
equity_curve = INITIAL_CAPITAL * (1 + portfolio_returns).cumprod()

# 等权基准 (所有20只等权)
ew_returns = returns_df.iloc[lookback:].mean(axis=1)
equity_ew = INITIAL_CAPITAL * (1 + ew_returns).cumprod()

print(f'调仓次数: {len(selection_history)}')
print(f'策略终端净值: {equity_curve.iloc[-1]:,.0f}')
print(f'等权终端净值: {equity_ew.iloc[-1]:,.0f}')
print(f'\n最新持仓: {current_holdings}')

In [ ]:
# Cell 8: 回测统计

bench_returns = None
if bench_close is not None:
    bench_close_aligned = bench_close.reindex(portfolio_returns.index).ffill()
    bench_returns = bench_close_aligned.pct_change().fillna(0)

metrics_screening = summary_table(portfolio_returns, equity_curve, bench_returns)
metrics_ew = summary_table(ew_returns, equity_ew, bench_returns)

compare_keys = ['年化收益率', '年化波动率', '夏普比率', '最大回撤', 'Calmar比率', '胜率']
print(f'{"指标":<12} {"筛选策略":<14} {"等权基准":<14}')
print('-' * 42)
for k in compare_keys:
    print(f'{k:<12} {metrics_screening.get(k, "N/A"):<14} {metrics_ew.get(k, "N/A"):<14}')

# 筛选漏斗统计
sel_df = pd.DataFrame(selection_history)
print(f'\n=== 筛选漏斗统计 ===')
print(f'Stage1 平均通过: {sel_df["stage1_count"].mean():.1f} / {len(stock_names)}')
print(f'Stage2 平均通过: {sel_df["stage2_count"].mean():.1f}')
print(f'最终选股数: {top_n}')
print(f'平均换手率: {sel_df["turnover"].mean():.1%}')

In [ ]:
# Cell 9: 可视化
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 1) 策略对比
plot_cumulative_comparison(
    {'多阶段筛选': portfolio_returns, '等权基准': ew_returns},
    title='多阶段筛选策略 vs 等权基准'
)

# 2) 回撤
plot_drawdown(equity_curve, title='多阶段筛选策略 - 回撤')

# 3) 漏斗图 (静态)
fig, ax = plt.subplots(figsize=(10, 6))
stages = ['初始池', 'Stage1\n动量', 'Stage2\n波动率', 'Stage3\nTop-5']
avg_counts = [len(stock_names), sel_df['stage1_count'].mean(),
              sel_df['stage2_count'].mean(), top_n]
colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

bars = ax.barh(stages[::-1], avg_counts[::-1], color=colors_bar[::-1], height=0.6)
for bar, count in zip(bars, avg_counts[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{count:.0f}只', va='center', fontsize=12, fontweight='bold')
ax.set_title('平均筛选漏斗', fontsize=14, fontweight='bold')
ax.set_xlabel('股票数量')
plt.tight_layout()
plt.show()

# 4) 绩效表
plot_metrics_table(metrics_screening, title='多阶段筛选策略 绩效指标')

## 结果分析

### 多阶段筛选的优势
1. **逻辑透明**: 每个阶段的筛选标准清晰可解释，符合基金经理实际决策流程
2. **风险控制**: 波动率过滤阶段有效剔除了高风险股票
3. **动量效应**: 动量因子在A股市场中期(1-6个月)表现稳定
4. **换手率可控**: 层层筛选减少了不必要的换仓

### 改进方向
- 各阶段权重可通过历史回测优化
- 加入基本面筛选阶段 (PE/ROE等)
- 引入行业分散化约束，避免行业过度集中
- 考虑市值因子，在大小盘轮动时动态调整